In [13]:
from extract.shared.http import get_soup
from extract.shared.links import normalize_links, extract_product_url
from extract.shared.html_extractors import extract_json_block, extract_materials
from extract.sites.aym.adapter import build_product_dict
from extract.sites.aym.config import CONFIG


In [14]:
base_url=CONFIG["base_url"]
category_locators=CONFIG["category_locators"]
product_locator=CONFIG["product_locator"]
materials_locator=CONFIG["materials_locator"]

In [15]:
print(base_url)

https://www.aym-studio.com


In [16]:
category_links=normalize_links(base_url,category_locators)  
print(category_links)

['https://www.aym-studio.com/collections/all-bottoms', 'https://www.aym-studio.com/collections/dresses', 'https://www.aym-studio.com/collections/all-tops']


In [17]:
product_links=[]

for link in category_links:

   
    print("\n==============================")
    print("CATEGORY PAGE:", link)
    

    # --- fetch page ---
    soup=get_soup(link)

    # --- extract raw product URLs ---
    raw_urls=extract_product_url(soup, product_locator)[:3]

    print("RAW URLs:")
    print(raw_urls)

    # --- normalize URLs ---
    normalized_urls=normalize_links(base_url, raw_urls)

    print("NORMALIZED URLS:")
    print(normalized_urls)

    # --- store results ---
    for url in normalized_urls:
        product_links.append(url)

print("\n==============================")
print("TOTAL PRODUCT LINKS:", len(product_links))
print(product_links[:10])


CATEGORY PAGE: https://www.aym-studio.com/collections/all-bottoms
RAW URLs:
['/products/sammi-shorts-in-bamboo', '/products/short-drape-cape?variant=56042035904889&color=teal-green', '/products/fletcher-skirt']
NORMALIZED URLS:
['https://www.aym-studio.com/products/fletcher-skirt', 'https://www.aym-studio.com/products/short-drape-cape?variant=56042035904889&color=teal-green', 'https://www.aym-studio.com/products/sammi-shorts-in-bamboo']

CATEGORY PAGE: https://www.aym-studio.com/collections/dresses
RAW URLs:
['/products/short-drape-cape?variant=56042035904889&color=teal-green', '/products/audrey-midi-dress-in-organic-bamboo', '/products/cammie-dress']
NORMALIZED URLS:
['https://www.aym-studio.com/products/audrey-midi-dress-in-organic-bamboo', 'https://www.aym-studio.com/products/cammie-dress', 'https://www.aym-studio.com/products/short-drape-cape?variant=56042035904889&color=teal-green']

CATEGORY PAGE: https://www.aym-studio.com/collections/all-tops
RAW URLs:
['/products/long-drape-c

In [18]:
product_links=list(set(product_links))

products=[]

for link in product_links:

    product_soup=get_soup(link)
    
    
    print("LOCATOR:", materials_locator)
    print("TAG:", materials_locator.get("tag"))
    print("CLASS:", materials_locator.get("class"))

    json_data=extract_json_block(product_soup)
    materials=extract_materials(product_soup, materials_locator)

    if not json_data:
        print("!!! Missing JSON:", link)
        continue

    try:
        product=build_product_dict(json_data, materials)

        
        
        print(product)

        products.append(product)

    except Exception as e:
        print("!!! ERROR building product:", link)
        print(e)

print("\nDONE:", len(products), "products")

LOCATOR: {'tag': 'div', 'class': 'accordion__content prose'}
TAG: div
CLASS: accordion__content prose
{'brand': 'AYM', 'sub_category': 'Capes', 'product_name': 'Short Drape Cape', 'price': '49.00', 'currency': 'GBP', 'product_url': 'https://www.aym-studio.com/products/short-drape-cape?variant=56143980986745', 'materials_raw': '68% Bamboo Viscose, 28% Cotton, 4% Elastane.\nFabric country of origin: Turkey\nSoft by design:\nSoft by design:\nWe carefully select fabrics for their luxurious hand feel, softness, and smooth finish.\nScratch-free comfort:\nScratch-free comfort:\nThis garment is fully double-layered, so you’ll never feel a raw seam on the inside. All stitching is enclosed between the layers, creating a smooth, seamless interior that feels gentle against your skin.\nSculpted with stretch:\nSculpted with stretch:\nOur double-layer construction not only adds durability and structure, but also delivers a clean, elevated finish. The result is a smooth, sculpted silhouette from every